## Link to article

This notebook is included in the documentation, where the interactive Plotly charts show up. See:
https://pegasystems.github.io/pega-datascientist-tools/latest/articles/vf_analysis.html

> **📦 Optional dependencies**
>
> This article uses features from the pdstools `adm` extra. Install with your favorite package manager, e.g. `uv pip install "pdstools[adm]"`.


In [1]:
# These lines are only for rendering in the docs, and are hidden through Jupyter tags
# Do not run if you're running the notebook seperately

import plotly.io as pio

pio.renderers.default = "notebook_connected"

# Value Finder Analysis
Every Value Finder simulation populates a dataset, the **pyValueFinder** dataset.  This dataset contains a lot more information than is what is currently presented on screen.

The data held in this dataset can be analysed to uncover insights into your decision framework. This notebook provides a sample analysis of the Value Finder simulation results.

In the data folder we've stored a copy of such a dataset. By default, this notebook uses that sample. To use your own data, export the **pyValueFinder** dataset from Dev Studio and set `VALUE_FINDER_DATA_PATH` to the export file path.

This is how Value Finder results of the sample data are presented in Pega (8.6, it may look different in other versions):

![Pega value finder screen](pegarun_8_6_0.png)

For the sample provided, the relevant action setting is 1.2%. There are 10.000 customers, 3491 without actions, 555 with only irrelevant actions and 5954 with at least one relevant action.

PDSTools defines a class **ValueFinder** that wraps the operations on this dataset. The "datasets" import is used for the example but you won't need this if you load your own Value Finder dataset.

It is also possible to supply a dataframe as the `df` argument directly.

In [2]:
import os
from pathlib import Path
from pdstools import ValueFinder, datasets
import polars as pl
import numpy as np

# Set VALUE_FINDER_DATA_PATH to your exported pyValueFinder file.
value_finder_path = os.environ.get("VALUE_FINDER_DATA_PATH")
vf = (
    ValueFinder.from_ds_export(
        filename=Path(value_finder_path).name,
        base_path=Path(value_finder_path).parent,
    )
    if value_finder_path
    else datasets.sample_value_finder()
)

When reading the data, we filter out unnecessary data, and the result is kept in the `df` property:

In [3]:
vf.df.head(5).collect()

Treatment,DynamicTemplateName,Weight,SubjectID,Organization,ActionContext,Strategy,UpdateDateTime,SubjectHash,ModelEvidence,UpdateopName,ModelControlGroup,Bundleparent,ModelPerformance,Quantity,Direction,Istransactional,TemplateName,SpecifySMSMessage,WorkID,ImageURL,Channel,IsEmailEnabled,PreviousComponent,ActionLimit,Updateoperator,IsPropositionActive,DeliverOffline,ContextWeight,CustomerHash,Segment,Interaction,KeyCode,SubjectType,Rank,FinalPropensity,Weight_2,…,UniqueID,CustomerID,Unit,Benefits,IsSMSEnabled,InteractionID,Application,MaturityCap,DynamicTemplateIssue,TemplateType,EndDate,Division,Priority,StartingPropensity,Propensity,Lastchannel,Issue,Applyanalytics,ModelExecutionError,WhyRelevant,ApplicationVersion,Priority_2,Group,OriginalSubjectID,StartingEvidence,OptimizePredictionOutcome,Journey,Agentcompensation,Pricing,Name,IsWebEnabled,BundleName,EmailSubject,SpecifyEmailSubject,Communicateto,IsPaidEnabled,Cost
str,str,f64,str,str,str,str,i64,str,i64,str,str,bool,str,i64,cat,bool,str,bool,str,str,cat,bool,str,i64,str,str,bool,str,i64,str,str,str,cat,u16,f64,f64,…,str,str,str,str,bool,str,str,f64,str,str,str,str,f64,str,f64,str,cat,bool,bool,str,str,f64,cat,str,u32,bool,str,str,str,str,bool,str,str,bool,str,bool,f64
"""Deposit_SMSTreatment""","""ChannelAction_Template""",null,"""Customer-1""","""DMOrg""","""Customer""","""Opp_NBA_All""",1629803581090,"""0.4388105018174723""",96,"""Piyush Vashisht""","""Test""",false,"""0.5""",null,"""Outbound""",false,"""""",false,"""Opp_NBA_AlDF_Simulat-1""","""web/DepositAccountOffer.png""","""SMS""",true,"""Customers""",null,"""vashp""","""Always""",false,null,10,null,"""——""","""SDSC2""","""CDHSample-Data-Customer""",3,0.278077,null,…,null,"""Customer-1""","""Unit""","""Students under age 23 are elig…",true,"""6320879002230024255""","""CDHSample""",0.1768,"""Sales""","""""","""20181231T163800.000 GMT""","""Div""",null,"""0.5""",0.269231,null,"""Sales""",true,null,"""For students under age 23 are …","""01.01.01""",0.269231,"""DepositAccounts""","""Customer-1""",100,true,null,"""2""","""No monthly maintenance fee whe…","""StudentChecking""",null,null,null,null,null,null,null
"""Mobile_ExternalSMSTreatment""","""Usage_ChannelAction_Template""",null,"""Customer-100""","""DMOrg""","""Customer""","""Opp_NBA_All""",1629803581074,"""0.940830237245977""",0,"""Piyush Vashisht""","""Test""",false,"""0.5""",null,"""Outbound""",false,"""Mobile_SMSDBTemplate""",true,"""Opp_NBA_AlDF_Simulat-1""",null,"""SMS""",true,"""Customers""",null,"""vashp""","""Always""",true,null,1,null,"""——""","""UMGUMA1""","""CDHSample-Data-Customer""",3,0.713095,null,…,null,"""Customer-100""","""Unit""",null,true,"""6320879002230024184""","""CDHSample""",0.02,"""Usage""","""DB""",null,"""Div""",null,"""1""",0.5,null,"""Usage""",true,null,null,"""01.01.01""",0.5,"""Mobilebanking""","""Customer-100""",25,true,null,null,null,"""GetTheUMobileApp""",null,null,null,null,null,null,null
"""Recommendations_SMSTreatment""","""Collections_ChannelAction_Temp…",null,"""Customer-1000""","""DMOrg""","""Customer""","""Opp_NBA_All""",1629803581035,"""0.8026197582592971""",0,"""Piyush Vashisht""","""Test""",false,"""0.5""",null,"""Outbound""",false,"""""",false,"""Opp_NBA_AlDF_Simulat-1""",null,"""SMS""",null,"""Customers""",null,"""vashp""","""Always""",false,null,0,null,"""——""","""CRSAT1""","""CDHSample-Data-Customer""",3,0.421306,null,…,null,"""Customer-1000""","""Unit""",null,true,"""6320879002230023934""","""CDHSample""",0.02,"""Collections""","""""",null,"""Div""",null,"""1""",0.5,null,"""Collections""",true,null,null,"""01.01.01""",0.5,"""Recommendations""","""Customer-1000""",25,true,null,null,null,"""SetupAutopayToday""",true,null,null,null,null,null,null
"""Deposit_SMSTreatment""","""ChannelAction_Template""",null,"""Customer-10000""","""DMOrg""","""Customer""","""Opp_NBA_All""",1629803581030,"""0.2919499647730856""",96,"""Piyush Vashisht""","""Test""",false,"""0.5""",null,"""Outbound""",false,"""""",false,"""Opp_NBA_AlDF_Si

## Basic Analysis: Pie Charts and Customer Segmentation

The piechart shown in platform is based on a propensity threshold. For the sample data, this threshold follows from a propensity quantile of 5.2%.

The [`plot.pie_charts()`](https://pegasystems.github.io/pega-datascientist-tools/latest/autoapi/pdstools/valuefinder/Plots/index.html#pdstools.valuefinder.Plots.Plots.pie_charts) function shows piecharts for all stages in the engagement policies (in platform you only see the last one). The threshold is calculated automatically, but you can specify it explicitly.

In [4]:
# Automatic threshold calculation
vf.plot.pie_charts()

# Or specify threshold explicitly (e.g., 5.2% quantile)
vf.plot.pie_charts(quantiles=[0.052])

Hover over the charts to see the details. For the sample data, the rightmost pie chart corresponds to the numbers in Pega as shown in the screenshot above.

* Red = customers not receiving any action
* Yellow = customers not receiving any "relevant" actions, sometimes also called "under served"
* Green = customers that receive at least one "relevant" action, sometimes also called "well served"

With "relevant" defined as having a propensity above the threshold. This defaults to the 5th percentile.

## Propensity Analysis

Insights into the propensity distribution per stage are crucial. The [`plot.propensity_threshold()`](https://pegasystems.github.io/pega-datascientist-tools/latest/autoapi/pdstools/valuefinder/Plots/index.html#pdstools.valuefinder.Plots.Plots.propensity_threshold) function visualizes this distribution. You often see a spike at 0.5, which corresponds to models without responses (their propensity defaults to 0.5/1 = 0.5).

The dotted vertical line represents the computed threshold.

In [5]:
vf.plot.propensity_threshold()

These different propensities represent 

* *pyModelPropensity* = the actual propensities from the models
* *pyPropensity* = model or random propensity, depending on the ModelControl (or, when models are executed from an extension point after the standard Predictions, their propensity, but such a configuration is not supported by Value Finder)
* *FinalPropensity* = the propensity after possible adjustments by Thompson Sampling; Thompson Sampling basically smoothes the propensities, you would expect any peak at 0.5 caused by empty models to be smoothed out

We can also look at the propensity distributions across the different stages. This is based on the model propensities, not any of the subsequent overrides:

In [6]:
vf.plot.propensity_distribution()

## Threshold Simulation

The effect of the propensity threshold selection on the number of actions for a customer can be simulated by supplying a list of quantiles or propensities. This generates aggregated counts per stage for different threshold values.

In [7]:
# Simulate thresholds from 1% to 99% in 1% increments
vf.plot.pie_charts(quantiles=np.arange(0.01, 1, 0.01))

The further to the left you put the slider threshold, the more "green" you will see. As you raise the threshold, more customers will be reported as getting "not relevant" actions.

The same effect can be visualized as a funnel. Use [`plot.distribution_per_threshold()`](https://pegasystems.github.io/pega-datascientist-tools/latest/autoapi/pdstools/valuefinder/Plots/index.html#pdstools.valuefinder.Plots.Plots.distribution_per_threshold) to show the threshold on the x-axis.

In [8]:
# Default threshold range
vf.plot.distribution_per_threshold()

# Custom threshold range (1% to 99% in 1% increments)
vf.plot.distribution_per_threshold(quantiles=np.arange(0.01, 1, 0.01))

## Funnel Analysis

You can analyze how individual actions are distributed across stages. Since there are typically many actions, it's useful to filter to a specific group or issue.

The [`plot.funnel_chart()`](https://pegasystems.github.io/pega-datascientist-tools/latest/autoapi/pdstools/valuefinder/Plots/index.html#pdstools.valuefinder.Plots.Plots.funnel_chart) function shows how actions flow through the stages. **Rule of thumb**: If there are only a few actions in each stage, or if certain actions are completely filtered out between stages, this may indicate strong filtering issues.

In [9]:
# Filter to Sales issue actions (adjust query for your data)
vf.plot.funnel_chart("Name", query=pl.col("Issue") == "Sales")

### Hierarchical Funnel Views

Start with a coarse-grained view at the Issue level, then drill down to Group or individual Action level as needed.

In [10]:
# High-level view: Issue level
vf.plot.funnel_chart("Issue")

# Mid-level view: Groups within Sales issue
vf.plot.funnel_chart("Group", query=pl.col("Issue") == "Sales")